# TAE-IA · Module 6 · L06 — Schedulers, Speed, and Memory Optimization

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Lesson** | L06 |
| **Track** | A — Vision |
| **Estimated duration** | 2 hours |
| **GPU required** | T4 (Colab) |
| **Prerequisites** | L01–L05 completed |

## Learning objectives
By the end of this notebook you will be able to:
- [ ] Swap schedulers on a loaded pipeline and explain the quality/speed trade-off of each
- [ ] Measure peak VRAM usage before and after enabling attention slicing
- [ ] Load LCM-LoRA and generate images in 4 steps with correct CFG settings
- [ ] Choose the right scheduler and step count for a given latency budget

## Before you start
- L01-L05 completed
- T4 GPU runtime selected (`Runtime > Change runtime type > T4 GPU`)

---

## Cell 0 — Setup (always run this first)

> Mounts Drive, checks GPU, fixes seed, and logs in to HuggingFace.

In [ ]:
# ================================================================
# TAE-IA M6 · Standard Setup -- DO NOT MODIFY THIS CELL
# ================================================================
import os, sys, random, time, shutil
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

MODEL_CACHE = '/content/drive/MyDrive/TAE_IA_M6/models'   # persistent, survives sessions
LOCAL_CACHE = '/content/model_cache'                      # ephemeral, dies with the runtime
os.makedirs(MODEL_CACHE, exist_ok=True)
os.makedirs(LOCAL_CACHE, exist_ok=True)

# Clear any cache left by earlier course versions.
for _leftover in ('hub', 'xet'):
    _p = os.path.join(MODEL_CACHE, _leftover)
    if os.path.exists(_p):
        shutil.rmtree(_p)

# ----------------------------------------------------------------
# Model cache with automatic fallback
# ----------------------------------------------------------------
# Models are cached on Drive so they survive a session restart. A free
# Google account has only 15 GB, shared with Gmail and Photos, which is
# less than this course needs end to end. So when Drive has no room we
# cache on the runtime's own disk instead: the lab still runs, but that
# copy is deleted when the session ends.
#
# The choice is made per model, not once per notebook. A model already
# sitting on Drive keeps being read from Drive even when Drive is far
# too full to accept anything new.

DRIVE_HEADROOM_GB = 1.0    # never fill Drive to the last byte
SIZE_MARGIN       = 1.15   # temp files and metadata written during a download


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return 0.0


def _is_cached(path):
    """True if <path> holds anything but HuggingFace's incomplete-download folder."""
    return os.path.isdir(path) and any(e != '.cache' for e in os.listdir(path))


def _disk_full(exc):
    """ENOSPC, quota exceeded, or the I/O error drivefs raises when Drive is full."""
    return (getattr(exc, 'errno', None) in (28, 122, 5)
            or 'no space' in str(exc).lower()
            or 'quota' in str(exc).lower())


def cache_dir(name, needed_gb):
    """Return the directory model <name> should live in, preferring Drive."""
    drive_dir = os.path.join(MODEL_CACHE, name)
    local_dir = os.path.join(LOCAL_CACHE, name)

    if _is_cached(drive_dir):
        return drive_dir            # already on Drive: reading it costs no space
    if _is_cached(local_dir):
        return local_dir            # already fell back earlier this session

    required = needed_gb * SIZE_MARGIN + DRIVE_HEADROOM_GB
    free = _free_gb(MODEL_CACHE)
    if free >= required:
        os.makedirs(drive_dir, exist_ok=True)
        print(f'[cache] {name} -> Drive ({needed_gb:.2f} GB needed, {free:.1f} GB free).')
        return drive_dir

    local_free = _free_gb('/content')
    if local_free < required:
        raise RuntimeError(
            f'{name} needs ~{needed_gb:.1f} GB, but only {free:.1f} GB is free on '
            f'Drive and {local_free:.1f} GB on the runtime disk.\n'
            f'Free space in Google Drive (delete unused TAE_IA_M6/models subfolders) '
            f'and re-run this cell.')

    os.makedirs(local_dir, exist_ok=True)
    print(f'[cache] Not enough room on Drive for {name}: needs {needed_gb:.2f} GB\n'
          f'        plus margin, {free:.1f} GB free. Caching on the runtime instead.\n'
          f'        The lab runs normally, but this copy is deleted when the session\n'
          f'        ends and downloads again next time. Free space in Drive to avoid\n'
          f'        the repeat download.')
    return local_dir


def cached_fetch(name, needed_gb, download):
    """Run download(target_dir) in the best available cache and return its result.

    Retries on the runtime disk if Drive fills up mid-download: a pre-flight
    space check cannot catch a quota that runs out halfway through.
    """
    target = cache_dir(name, needed_gb)
    try:
        return download(target)
    except OSError as e:
        if not _disk_full(e) or target.startswith(LOCAL_CACHE):
            raise
        print(f'[cache] Drive ran out of room mid-download ({e}).\n'
              f'        Discarding the partial copy and retrying on the runtime disk.')
        shutil.rmtree(target, ignore_errors=True)
        fallback = os.path.join(LOCAL_CACHE, name)
        os.makedirs(fallback, exist_ok=True)
        return download(fallback)


def cached_snapshot(name, repo_id, needed_gb, **kwargs):
    """snapshot_download into the best available cache, returning its path."""
    from huggingface_hub import snapshot_download

    def _dl(target):
        snapshot_download(repo_id, local_dir=target, **kwargs)
        return target

    return cached_fetch(name, needed_gb, _dl)


_drive_free = _free_gb(MODEL_CACHE)
print(f'Model cache: {MODEL_CACHE}  ({_drive_free:.1f} GB free on Drive)')
if _drive_free < 1.0:
    print('WARNING: under 1 GB free on Drive. Models will cache on the runtime,\n'
          '         but saving your lab outputs may fail. Free space in Drive.')

import torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Go to: Runtime > Change runtime type > T4 GPU')
    raise SystemExit('GPU required.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f'Fixed seed: {SEED}')
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')

# HuggingFace login -- needed every new Colab session
import huggingface_hub

try:
    _token = huggingface_hub.get_token()
except Exception:
    _token = None

if _token:
    print(f"Already logged in to HuggingFace as: {huggingface_hub.whoami()['name']}")
else:
    try:
        from google.colab import userdata
        _hf_token = userdata.get('HF_TOKEN')
    except Exception:
        _hf_token = None

    if _hf_token:
        huggingface_hub.login(token=_hf_token, add_to_git_credential=False)
        print(f"Logged in to HuggingFace as: {huggingface_hub.whoami()['name']}")
    else:
        raise RuntimeError(
            'No HF_TOKEN found in Colab Secrets (key icon, left sidebar).\n'
            'Add a secret named HF_TOKEN with your HuggingFace read token, enable notebook access, '
            'then re-run this cell.\n'
            'See L00 Cell 4 if you need to generate a token or accept the SD 1.5 license.'
        )

In [ ]:
# ================================================================
# Install dependencies for L06
# ================================================================
!pip install diffusers transformers accelerate peft -q

# Colab preinstalls torchao 0.10, and current peft refuses to run alongside
# any torchao older than 0.16 -- it raises ImportError the moment LoRA
# weights are loaded (Section 2.4). Nothing in this lab uses torchao, so
# removing it is safer than upgrading it: a newer torchao can pull a
# different torch build and force a runtime restart in the middle of the lab.
!pip uninstall -y -q torchao

import importlib
importlib.invalidate_caches()   # so peft stops finding the removed package

import diffusers, peft
print(f'diffusers {diffusers.__version__}  |  peft {peft.__version__}')


---
## Part 1 — Context and Key Concepts

> Read this before running any code.

### What is a scheduler?

During inference, the UNet iteratively removes noise from a latent. At each step it predicts the slightly-less-noisy version. The **scheduler** defines:
- The noise levels at each timestep (the noise schedule)
- How to use the UNet's prediction to move from a noisy latent to a less-noisy one (the update rule)

All schedulers use the **same frozen UNet weights**. Swapping schedulers is a one-line change that requires no weight reloading.

### Why schedulers differ in convergence speed

First-order solvers (PNDM, DDIM) make a linear approximation of the denoising trajectory at each step — accurate but slow to converge. Higher-order solvers (DPM++ 2M, UniPC) use 2nd-order approximations that need fewer steps to reach the same quality.

| Scheduler | Order | Min. steps | Deterministic? |
|---|---|---|---|
| PNDM | Pseudo-numerical (≈2nd) | 20–25 | Yes |
| DDIM | 1st-order ODE | 20–30 | Yes |
| DPM++ 2M | 2nd-order ODE | 15–20 | Yes |
| UniPC | Predictor-corrector | 15–20 | Yes |

### Memory optimization options

- **`enable_attention_slicing()`** — processes self-attention in chunks instead of all at once. Reduces peak VRAM 15–30% at a ~10% speed cost. Recommended when running multiple models in sequence.
- **`enable_model_cpu_offload()`** — moves sub-models (VAE, text encoder) to RAM when not in use, only loading them to GPU for their specific step. Reduces VRAM to ~3 GB at a moderate speed cost.
- **`enable_sequential_cpu_offload()`** — moves individual *layers* to RAM. Maximum VRAM savings (~1 GB) but 3–5× slower. Use only as a last resort.

### LCM-LoRA

LCM (Latent Consistency Models) LoRA is a small adapter (~300 MB) trained with consistency distillation. It modifies SD 1.5 to converge in 4–8 steps instead of 20–50. Two key constraints:

1. **Must use `LCMScheduler`** — other schedulers produce poor results with this adapter
2. **`guidance_scale` must be low (0–2)** — the distillation process embeds CFG; high values cause oversaturation

The adapter is loaded on top of the cached SD 1.5 weights — no new base model download.

---

## Part 2 — Lab

### Section 2.0 — Load the SD 1.5 pipeline

In [ ]:
# Section 2.0 — Load the SD 1.5 pipeline
from diffusers import StableDiffusionPipeline
import torch, time, os
import matplotlib.pyplot as plt

OUTPUT_DIR = '/content/drive/MyDrive/TAE_IA_M6/L06_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fetch only the fp16 weights + configs (~2.7 GB) — shared with L01-L05 if already cached.
SD15_DIR = cached_snapshot(
    'sd15-local',
    "runwayml/stable-diffusion-v1-5",
    needed_gb=2.74,
    allow_patterns=["*.json", "*.txt", "*.fp16.safetensors"],
)

pipe = StableDiffusionPipeline.from_pretrained(
    SD15_DIR,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")
print(f"Pipeline loaded. Default scheduler: {type(pipe.scheduler).__name__}")


### Section 2.1 — Benchmark helper (warm-up + timing rules)

In [ ]:
# Section 2.1 — Benchmark helper
# Every timing in this lab goes through run(), so the measurement rules live in one place.

def gen(seed=SEED):
    return torch.Generator("cuda").manual_seed(seed)

BENCHMARK_PROMPT = "a mountain village in autumn, photorealistic, high detail, warm light"

def run(pipe, steps, cfg=7.5, label=""):
    """Run inference and return (image, elapsed_seconds, peak_vram_GB)."""
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0  = time.time()
    img = pipe(BENCHMARK_PROMPT, num_inference_steps=steps,
               guidance_scale=cfg, generator=gen()).images[0]
    torch.cuda.synchronize()
    elapsed  = time.time() - t0
    peak_gb  = torch.cuda.max_memory_allocated(0) / 1e9
    if label:
        print(f"{label:22s} | steps={steps:2d} | {elapsed:.1f}s | peak VRAM: {peak_gb:.2f} GB")
    return img, elapsed, peak_gb


def warmup(pipe, steps=4, cfg=7.5):
    """Burn one short generation so that one-time CUDA/cuDNN autotuning is not
    charged to whichever configuration happens to be measured first.

    Without this, the first row of every table below is a few seconds slower
    than it should be for reasons that have nothing to do with the scheduler
    being tested — which is exactly the kind of artefact that makes a
    benchmark lie."""
    _ = pipe(BENCHMARK_PROMPT, num_inference_steps=steps,
             guidance_scale=cfg, generator=gen()).images[0]
    torch.cuda.synchronize()

warmup(pipe)
print("Warm-up done — timings below exclude one-time CUDA setup cost.")


### Section 2.2 — Scheduler benchmark: 4 schedulers × 20 and 50 steps

In [ ]:
# Section 2.2 — Scheduler benchmark
from diffusers import (
    PNDMScheduler,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    UniPCMultistepScheduler,
)

base_config = pipe.scheduler.config

schedulers = {
    "PNDM":    PNDMScheduler,
    "DDIM":    DDIMScheduler,
    "DPM++2M": DPMSolverMultistepScheduler,
    "UniPC":   UniPCMultistepScheduler,
}
step_counts = [20, 50]

results = {}   # (scheduler_name, steps) -> (image, time, vram)
print(f"{'Scheduler':<22} | {'Steps':>5} | {'Time':>6} | {'Peak VRAM':>10}")
print("-" * 55)

for name, SchedClass in schedulers.items():
    pipe.scheduler = SchedClass.from_config(base_config)
    for steps in step_counts:
        img, t, vram = run(pipe, steps, label=f"{name} {steps}steps")
        img.save(os.path.join(OUTPUT_DIR, f"{name.replace('++','pp')}_{steps}steps.png"))
        results[(name, steps)] = (img, t, vram)

# Display as 4×2 grid
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for col, name in enumerate(schedulers.keys()):
    for row, steps in enumerate(step_counts):
        img, t, _ = results[(name, steps)]
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{name}\n{steps} steps · {t:.1f}s", fontsize=9)
        axes[row, col].axis('off')
plt.suptitle("Scheduler benchmark — same prompt and seed", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "scheduler_benchmark.png"), dpi=100)
plt.show()

**What do you observe?**  
- At 20 steps, which scheduler produced the highest perceived quality?
- For which scheduler was the quality jump from 20→50 steps most noticeable?
- Were the timing differences between schedulers significant?

*Write your observation here:*

(double-click to edit)

### Section 2.3 — Attention slicing: VRAM and speed trade-off

In [ ]:
# Section 2.3 — Attention slicing VRAM measurement
from diffusers import DPMSolverMultistepScheduler

# Reset to DPM++ 2M at 20 steps (best scheduler from benchmark)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(base_config)

# Baseline — no optimization
img_base, t_base, vram_base = run(pipe, steps=20, label="Baseline")

# With attention slicing
pipe.enable_attention_slicing()
img_sliced, t_sliced, vram_sliced = run(pipe, steps=20, label="Attention slicing")
pipe.disable_attention_slicing()   # reset for fair comparison

# With model CPU offload
pipe.enable_model_cpu_offload()
img_offload, t_offload, vram_offload = run(pipe, steps=20, label="Model CPU offload")

def vram_delta(base, other):
    """Report a peak-VRAM change honestly. Sub-MB float noise is not a saving,
    and printing it as "-0 MB" makes a real result look like a broken cell."""
    mb = (base - other) * 1024
    if abs(mb) < 1:
        return "no measurable change"
    return f"{mb:.0f} MB saved" if mb > 0 else f"{-mb:.0f} MB MORE used"


print("\nSummary:")
print(f"  Attention slicing:  {vram_delta(vram_base, vram_sliced):<22s} |  speed cost: +{t_sliced - t_base:.1f}s")
print(f"  Model CPU offload:  {vram_delta(vram_base, vram_offload):<22s} |  speed cost: +{t_offload - t_base:.1f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (title, img, t, vram) in zip(axes, [
    ("Baseline", img_base, t_base, vram_base),
    ("Attention slicing", img_sliced, t_sliced, vram_sliced),
    ("Model CPU offload", img_offload, t_offload, vram_offload),
]):
    ax.imshow(img)
    ax.set_title(f"{title}\n{t:.1f}s · {vram:.2f} GB VRAM", fontsize=10)
    ax.axis('off')
plt.suptitle("Memory optimization comparison — DPM++ 2M, 20 steps", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "memory_optimization.png"), dpi=100)
plt.show()

**What do you observe?**  
- How many MB did attention slicing save? Was the image quality affected?
- How much slower was model CPU offload? For what scenario would that trade-off be worth it?
- Were the three output images visually identical?
- If one of these techniques saved no memory at all, is that a bug in the lab or a fact about this GPU and this version of PyTorch? How would you tell the difference?

*Write your observation here:*

(double-click to edit)

### Section 2.4 — LCM-LoRA: 4-step generation

Load the LCM-LoRA adapter on top of the cached SD 1.5 weights and compare against standard generation.

In [ ]:
# Section 2.4 — LCM-LoRA
import gc
from diffusers import LCMScheduler

# Reload a clean pipeline for LCM (model_cpu_offload changes device map).
# Both loads below read from the already-cached local files -- no re-download.
del pipe; gc.collect(); torch.cuda.empty_cache()

pipe_lcm = StableDiffusionPipeline.from_pretrained(
    SD15_DIR,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

# ~135 MB, single file, not gated.
LCM_LORA_DIR = cached_snapshot(
    'lcm-lora-sdv1-5-local',
    "latent-consistency/lcm-lora-sdv1-5",
    needed_gb=0.13,
    allow_patterns=["*.safetensors"],
)
pipe_lcm.load_lora_weights(LCM_LORA_DIR)
pipe_lcm.fuse_lora()   # merges adapter weights into the base model
pipe_lcm.scheduler = LCMScheduler.from_config(pipe_lcm.scheduler.config)
print("LCM-LoRA loaded and fused.")

# Also load a standard DPM++ pipeline for comparison -- fuse_lora() above
# permanently mutated pipe_lcm's UNet, so this needs a separate weight copy
# rather than sharing .components the way L02-L05 do.
pipe_std = StableDiffusionPipeline.from_pretrained(
    SD15_DIR,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")
pipe_std.scheduler = DPMSolverMultistepScheduler.from_config(pipe_std.scheduler.config)
print("Standard DPM++ pipeline loaded.")

In [ ]:
# Section 2.4 — Generate comparison: LCM-LoRA vs. standard DPM++
# NOTE: LCM-LoRA requires guidance_scale=1.0 (high values cause oversaturation)

configs = [
    ("LCM-LoRA  4 steps",  pipe_lcm, 4,  1.0),
    ("LCM-LoRA  8 steps",  pipe_lcm, 8,  1.0),
    ("DPM++2M  20 steps",  pipe_std, 20, 7.5),
    ("DPM++2M  50 steps",  pipe_std, 50, 7.5),
]

# Both pipelines were built moments ago, so neither has paid its one-time CUDA
# autotune cost yet. Warm each on the path it will actually be measured on --
# LCM runs at guidance_scale=1.0, which skips classifier-free guidance entirely
# and therefore exercises different kernels than pipe_std does. Without this,
# the first row absorbs the setup cost and 4 steps and 8 steps come out almost
# the same, which is the exact artefact Section 2.1 warns about.
warmup(pipe_lcm, steps=4, cfg=1.0)
warmup(pipe_std, steps=4)

lcm_results = []
for label, p, steps, cfg in configs:
    img, t, _ = run(p, steps, cfg=cfg, label=label)
    img.save(os.path.join(OUTPUT_DIR, f"lcm_{label.replace(' ','_')}.png"))
    lcm_results.append((label, img, t))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (label, img, t) in zip(axes, lcm_results):
    ax.imshow(img)
    ax.set_title(f"{label}\n{t:.1f}s", fontsize=9)
    ax.axis('off')
plt.suptitle("LCM-LoRA vs. DPM++ 2M — same prompt and seed", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "lcm_comparison.png"), dpi=100)
plt.show()

**What do you observe?**  
- At 4 steps, how does LCM-LoRA quality compare to DPM++ at 20 steps?
- Is there a visible quality difference between LCM-LoRA at 4 vs. 8 steps?
- What specific quality differences do you notice (sharpness, color accuracy, fine detail)?

*Write your observation here:*

(double-click to edit)

---
## Part 3 — Exercises

### Exercise 1 — Find the step sweet spot for DPM++ 2M

**Task:** Using DPM++ 2M, run the same prompt at `steps = 5, 10, 15, 20, 30`. Display all five results in a grid. In a markdown cell, state at what step count quality stops improving and what specific feature (sharpness, fine detail, color) converges last.

**Expected output:** 5-image grid with step count and timing labels.

In [ ]:
# Exercise 1 -- DPM++ 2M step sweep
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise1.png"))
# ...

*At what step count does quality converge? What feature converges last?*

(double-click to edit)

### Exercise 2 — LCM-LoRA with wrong CFG

**Task:** Run LCM-LoRA at 4 steps with `guidance_scale = 1.0, 4.0, 7.5, 12.0`. Show all four results. In a markdown cell: at what CFG value does the output start to degrade, and how does it degrade (oversaturation, artifacts, color shift)?

**Expected output:** 4-image grid with CFG labels. This exercise documents *why* LCM-LoRA requires low CFG.

In [ ]:
# Exercise 2 -- LCM-LoRA CFG sensitivity
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise2.png"))
cfg_values = [1.0, 4.0, 7.5, 12.0]
# ...

*At what CFG does quality degrade, and how does it look?*

(double-click to edit)

---
## Part 4 — Critical Analysis

> Required. Answer with real outputs from today's session.

**4.1 — Best scheduler at 20 steps: which of the four schedulers produced the highest perceived quality? Identify one specific quality dimension (sharpness, color, fine detail, composition) where it clearly outperformed the others.**

*Write here (reference your Section 2.2 benchmark grid):*


---

**4.2 — Diminishing returns: for the best-performing scheduler, at what step count did quality stop improving noticeably? How does this compare with what you measured in L01 using PNDM?**

*Write here (reference your Exercise 1 results):*


---

**4.3 — VRAM trade-off: how many MB did attention slicing save, and how many seconds did it cost per image? Describe a real project scenario where paying that speed cost would be the right engineering decision.**

*Write here (reference your Section 2.3 measurements):*


---

**4.4 — LCM-LoRA use case: name a real application where LCM-LoRA at 4 steps (~0.7s) would be the right choice over DPM++ at 20 steps (~3s). What specific property of your application makes the quality trade-off acceptable?**

*Write here (be specific — e.g., "real-time preview thumbnails in a design tool", "mobile app with limited compute budget"):*


---
## Submission Checklist

- [ ] All cells ran from start to finish without errors
- [ ] Section 2.2 scheduler benchmark grid saved (`scheduler_benchmark.png`)
- [ ] Section 2.3 memory optimization comparison saved (`memory_optimization.png`)
- [ ] Section 2.4 LCM-LoRA comparison saved (`lcm_comparison.png`)
- [ ] Exercise 1 — DPM++ 2M step sweep with convergence analysis
- [ ] Exercise 2 — LCM-LoRA CFG sensitivity grid with explanation
- [ ] Part 4 — Critical Analysis completed (all 4 questions with real evidence)
- [ ] All outputs saved to `TAE_IA_M6/L06_output/` on Drive

**Save:** `File > Save a copy in Drive`

> **No Drive cleanup needed after this lesson** — SD 1.5 is still needed for L07, L08, and L11. The LCM-LoRA adapter (~135 MB) is negligible.

---
## Before You Close This Tab

- [ ] Confirmed all outputs from this session are saved in `TAE_IA_M6/L06_output/` on Drive (see checklist above)
- [ ] Disconnected and deleted this runtime: `Runtime > Disconnect and delete runtime`

Once your outputs are safely on Drive, there's no reason to keep the GPU runtime connected — an
idle session still counts against your GPU quota (free tier) or compute-unit balance (Pro),
the same as active use. Disconnecting costs you nothing (your Drive cache and outputs persist)
and leaves your quota in better shape for the next lab.

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L06*  
*Platform: Google Colab (T4 GPU) · Python 3.10*